<div style='background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%); padding: 44px 48px 38px; border-radius: 14px; margin-bottom: 4px;'>
  <h1 style='color: #ffffff; margin: 0 0 10px; font-size: 2.3em; letter-spacing: -0.5px; font-weight: 700;'>
    MYH Application Data , PostgreSQL &amp; FastAPI
  </h1>
  <p style='color: #b0c8de; margin: 0; font-size: 1.05em; line-height: 1.9;'>
    Steps 9 and 10 of the MYH data pipeline: schema creation, bulk load from curated CSV, live verification, and REST API.<br>
    <strong style='color: #7ec8e3;'>data_pipeline_de25 &nbsp;&middot;&nbsp; 9,983 rows &nbsp;&middot;&nbsp; 26 columns &nbsp;&middot;&nbsp; MYH Data Pipeline Project</strong>
  </p>
</div>

## Notebook contents

| # | Section | What it covers |
|:--|:--------|:---------------|
| 1 | Schema | DDL, column types, indexes, and design decisions |
| 2 | Load | Bulk-load the curated CSV into the `applications` table |
| 3 | Verification | Row counts, null audit, and value distributions |
| 4 | Notes | Idempotency, standalone script, and next steps |
| 5 | FastAPI | REST endpoints: models, query layer, and route definitions |
| 6 | Pipeline report | Automated pass/fail checks for all steps |

## Overview

This notebook covers the database and API layers of the pipeline.
The curated dataset produced by `data_preparation.ipynb` is loaded into PostgreSQL and exposed via a FastAPI application.

| Step | Description | Output |
|:---|:---|:---|
| 9 | Schema creation, bulk load from CSV, live verification | `applications` table - 9,983 rows |
| 10 | FastAPI - REST endpoints over the `applications` table | `api/` - 17 documented endpoints |

### Prerequisites

`data_preparation.ipynb` must have been run first - it produces `data/curated_applications.csv`.
Credentials are read from `.env` (copy `.env.example` and fill in your values).

---
## Step 9: PostgreSQL Database

The curated CSV exported from `data_preparation.ipynb` is loaded into a local PostgreSQL database (`data_pipeline_de25`).
This step covers three phases: **schema creation**, **bulk load**, and **live verification**.

**Why PostgreSQL?**

- The tabular structure of MYH application data maps naturally to a single relational table
- SQL aggregations (approval rates by year, region, education area) are concise and fast against an indexed table
- `psycopg2-binary` provides a pooled, direct connection to PostgreSQL used by both the notebook and the FastAPI layer in Step 10

**Architecture: single flat table, no ORM**

All 9 983 rows live in one flat `applications` table — no foreign keys, no normalisation.
The query patterns (filter + aggregate) do not benefit from joins.
psycopg2 `ThreadedConnectionPool` manages connection pooling; all queries are raw SQL with `%(name)s` parameters.

**Idempotency**

`DROP TABLE IF EXISTS` at the top of the schema means this notebook can be re-run at any time after a data update without manual cleanup.
The FastAPI `/refresh` endpoint replicates the same reload without Jupyter.


In [1]:
import os
from pathlib import Path

import pandas as pd
import psycopg2
import psycopg2.extras
from dotenv import load_dotenv
from IPython.display import HTML, display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer, SqlLexer

load_dotenv()

BASE = Path.cwd()
CSV_PATH = BASE / "data" / "curated_applications.csv"
SCHEMA_PATH = BASE / "db" / "schema.sql"

PG_DSN = dict(
    host=os.getenv("PG_HOST", "localhost"),
    port=int(os.getenv("PG_PORT", "5432")),
    dbname=os.getenv("PG_DATABASE", "data_pipeline_de25"),
    user=os.getenv("PG_USER", "postgres"),
    password=os.getenv("PG_PASSWORD", ""),
)

print(f"Database : {PG_DSN['dbname']}")
print(f"Host     : {PG_DSN['host']}:{PG_DSN['port']}")
print(f"CSV      : {CSV_PATH}")

Database : data_pipeline_de25
Host     : localhost:5432
CSV      : myh-data-pipeline\data\curated_applications.csv


In [2]:
import re

# ── Pipeline event log ───────────────────────────────────────────────────────────────────
PIPELINE_LOG: list[dict] = []


def log_event(step: str, level: str, message: str, detail: str = "") -> None:
    """Append one event to the pipeline log.

    level: "ok" | "warning" | "error"
    """
    PIPELINE_LOG.append(
        {"Step": step, "Level": level, "Message": message, "Detail": detail}
    )


# ── Setup checks ──────────────────────────────────────────────────────────────────
setup_ok = True

if not CSV_PATH.exists():
    log_event(
        "setup",
        "error",
        "CSV not found - run data_preparation.ipynb first",
        str(CSV_PATH),
    )
    setup_ok = False
else:
    log_event("setup", "ok", "CSV found", str(CSV_PATH))

if not SCHEMA_PATH.exists():
    log_event("setup", "error", "schema.sql not found", str(SCHEMA_PATH))
    setup_ok = False
else:
    log_event("setup", "ok", "schema.sql found", str(SCHEMA_PATH))

# ── Database creation (local only) ─────────────────────────────────────────────────
# Cloud databases (Neon, RDS, etc.) are pre-provisioned - skip creation there.
pg_host = PG_DSN["host"]
target_db = PG_DSN["dbname"]
local_hosts = {"localhost", "127.0.0.1"}

if pg_host in local_hosts:
    if not re.fullmatch(r"[A-Za-z0-9_]+", target_db):
        log_event(
            "setup", "error", f"Unsafe database name in PG_DATABASE: '{target_db}'"
        )
        setup_ok = False
    else:
        maintenance_dsn = {**PG_DSN, "dbname": "postgres"}
        try:
            # autocommit required - CREATE DATABASE cannot run inside a transaction.
            conn = psycopg2.connect(**maintenance_dsn)
            conn.autocommit = True
            try:
                with conn.cursor() as cur:
                    cur.execute(
                        "SELECT 1 FROM pg_database WHERE datname = %s",
                        (target_db,),
                    )
                    if cur.fetchone():
                        log_event(
                            "setup", "ok", f"Database '{target_db}' already exists"
                        )
                    else:
                        cur.execute(f'CREATE DATABASE "{target_db}"')
                        log_event("setup", "ok", f"Created database '{target_db}'")
            finally:
                conn.close()
        except Exception as exc:
            log_event("setup", "error", "Could not verify or create database", str(exc))
            setup_ok = False
else:
    log_event(
        "setup",
        "ok",
        f"Cloud host detected ({pg_host}) - skipping database creation check",
    )

# ── Connectivity check ─────────────────────────────────────────────────────────────────
try:
    conn = psycopg2.connect(**PG_DSN)
    conn.close()
    log_event("setup", "ok", f"Database reachable: {target_db}")
except Exception as exc:
    log_event("setup", "error", "Cannot connect to database", str(exc))
    setup_ok = False

print(f"Setup: {'OK' if setup_ok else 'ERRORS - see Pipeline Report for details'}")

Setup: OK


---
## 1. Schema

The table mirrors the canonical schema from `data_preparation.ipynb` exactly.
Every column in the CSV maps to a PostgreSQL column , no transformations happen at load time.

In [3]:
# Read schema.sql and render it with SQL syntax highlighting.
# pygments is already installed as a Jupyter dependency , no extra package needed.
schema_sql = SCHEMA_PATH.read_text(encoding="utf-8")

formatter = HtmlFormatter(style="monokai", noclasses=True)
highlighted_sql = highlight(schema_sql, SqlLexer(), formatter)

# The monokai theme default background (#272822) is brownish-grey; replace it
# with the dark blue used throughout this notebook for visual consistency.
highlighted_sql = highlighted_sql.replace("background: #272822", "background: #1a2535")

display(HTML(highlighted_sql))

### Design decisions

| Decision | Reasoning |
|:---|:---|
| `SERIAL PRIMARY KEY` | `diarienummer` is not unique within a year , one application covering several municipalities appears as multiple rows (2019 source data). A synthetic key avoids this constraint. |
| `SMALLINT` for numerics | `yh_poang`, `studietakt_procent`, `antal_kommuner`, `sokta_omgangar`, `beviljade_omgangar` all fit within ±32,767 , halves storage vs `INTEGER`. |
| `BOOLEAN` for flags | `is_distance` and `has_multiple_municipalities` are always true/false , no ambiguous strings in the DB. |
| Nullable columns | `typ_av_examen` (absent 2018/2020), SUN5/SeQF (absent pre-2023), round counts (absent 2018–2019) , nulls document the source limitation, not a pipeline error. |
| 5 indexes | `source_year`, `beslut_normalized`, `lan`, `utbildningsomrade`, `utbildningsanordnare` , the primary filter and group-by columns expected by the API. |

<div style='border-left: 4px solid #8e6bbf; padding: 12px 16px; border-radius: 4px; margin: 16px 0; background: rgba(142, 107, 191, 0.1);'>
<strong>No UNIQUE constraint on (diarienummer, source_year):</strong> A single application in 2019 can legitimately appear on multiple rows , one per study municipality. The 2019 <code>Lista</code> sheet records location-level granularity rather than application-level. The <code>id</code> column is the only row-level key.
</div>

In [4]:
print("Applying schema ...")

try:
    conn = psycopg2.connect(**PG_DSN)
    try:
        with conn.cursor() as cur:
            for statement in schema_sql.split(";"):
                statement = statement.strip()
                if statement:
                    cur.execute(statement)
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()
    log_event("schema", "ok", "Schema applied (DROP + CREATE + indexes)")
except Exception as exc:
    log_event("schema", "error", "Failed to apply schema", str(exc))
    raise

try:
    conn = psycopg2.connect(**PG_DSN)
    try:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT COUNT(*) FROM information_schema.columns "
                "WHERE table_name = 'applications' AND table_schema = 'public'"
            )
            column_count = cur.fetchone()[0]
            cur.execute(
                "SELECT COUNT(*) FROM pg_indexes "
                "WHERE tablename = 'applications' AND schemaname = 'public'"
            )
            index_count = cur.fetchone()[0]
    finally:
        conn.close()
    log_event(
        "schema", "ok", f"Verified: {column_count} columns, {index_count} indexes"
    )
except Exception as exc:
    log_event("schema", "error", "Schema verification query failed", str(exc))
    raise

print(f"Table 'applications' created - {column_count} columns, {index_count} indexes")

Applying schema ...
Table 'applications' created - 27 columns, 6 indexes


---
## 2. Load

The CSV is read into a DataFrame, integer columns are cast to standard Python numerics
(pandas writes `Int64` nullable integers as strings in CSV), and boolean columns are
mapped from string `"True"/"False"` to real Python bools before the bulk insert.

In [ ]:
try:
    # Read the curated CSV produced by data_preparation.ipynb.
    # utf-8-sig strips the BOM written by pandas' to_csv(encoding="utf-8-sig").
    applications = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
    log_event(
        "load",
        "ok",
        f"CSV read: {len(applications):,} rows, {len(applications.columns)} columns",
    )
except FileNotFoundError:
    log_event(
        "load",
        "error",
        "CSV not found - run data_preparation.ipynb first",
        str(CSV_PATH),
    )
    raise
except Exception as exc:
    log_event("load", "error", "Failed to read CSV", str(exc))
    raise

# pandas writes nullable Int64 columns as plain strings in CSV (e.g. "320", "nan").
# pd.to_numeric converts them back to float/int, coercing leftover "nan" to NaN,
# which psycopg2 then maps to NULL in the SMALLINT columns.
integer_columns = [
    "yh_poang",
    "studietakt_procent",
    "antal_kommuner",
    "sokta_omgangar",
    "beviljade_omgangar",
]
for column in integer_columns:
    if column not in applications.columns:
        log_event("load", "error", f"{column}: column missing from CSV")
        continue
    before_nulls = int(applications[column].isnull().sum())
    applications[column] = pd.to_numeric(applications[column], errors="coerce")
    new_nulls = int(applications[column].isnull().sum()) - before_nulls
    if new_nulls > 0:
        log_event(
            "load",
            "warning",
            f"{column}: {new_nulls} value(s) could not be coerced to numeric",
        )

# CSV round-trips Python bools as the strings "True" / "False".
# Map them back so psycopg2 inserts real BOOLEAN values instead of text.
boolean_columns = ["is_distance", "has_multiple_municipalities"]
for column in boolean_columns:
    if column not in applications.columns:
        log_event("load", "error", f"{column}: column missing from CSV")
        continue
    applications[column] = applications[column].map(
        {"True": True, "False": False, True: True, False: False}
    )

dtype_summary = pd.DataFrame(
    {
        "dtype": applications.dtypes,
        "non-null": applications.count(),
        "null": applications.isnull().sum(),
    }
).rename_axis("Column")

display(
    dtype_summary.style.set_caption(
        "Table 1 - Column types and null counts after CSV read"
    ).set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("font-size", "0.95em"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("padding-bottom", "8px"),
                ],
            }
        ]
    )
)
print(f"Shape: {applications.shape[0]:,} rows x {applications.shape[1]} columns")

In [6]:
print("Inserting rows ...")

try:
    columns = list(applications.columns)
    rows = [
        tuple(None if pd.isna(v) else v for v in row)
        for row in applications.itertuples(index=False, name=None)
    ]
    conn = psycopg2.connect(**PG_DSN)
    try:
        with conn.cursor() as cur:
            # execute_values sends rows in page_size batches - faster than
            # executemany and avoids the parameter-count limit of plain INSERT.
            psycopg2.extras.execute_values(
                cur,
                "INSERT INTO applications (" + ", ".join(columns) + ") VALUES %s",
                rows,
                page_size=500,
            )
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()
    log_event("insert", "ok", f"Bulk insert complete: {len(applications):,} rows")
    print(f"Insert complete - {len(applications):,} rows written")
except Exception as exc:
    log_event("insert", "error", "Bulk insert failed", str(exc))
    raise

Inserting rows ...
Insert complete - 9,983 rows written


---
## 3. Verification

Three checks against the live database: row count and approval rate per year,
a null audit on core non-nullable columns, and value distribution for the two
most important categorical fields.

In [ ]:
# Query the live table - row counts and approval rates per year.
conn = psycopg2.connect(**PG_DSN)
try:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                source_year AS "Year",
                COUNT(*) AS "Rows",
                SUM(CASE WHEN beslut_normalized = 'approved'  THEN 1 ELSE 0 END) AS "Approved",
                SUM(CASE WHEN beslut_normalized = 'rejected'  THEN 1 ELSE 0 END) AS "Rejected",
                SUM(CASE WHEN beslut_normalized = 'withdrawn' THEN 1 ELSE 0 END) AS "Withdrawn",
                ROUND(
                    100.0 * SUM(CASE WHEN beslut_normalized = 'approved' THEN 1 ELSE 0 END)
                    / COUNT(*), 1
                ) AS "Approval %"
            FROM applications
            GROUP BY source_year
            ORDER BY source_year
        """)
        year_statistics = pd.DataFrame(
            cur.fetchall(),
            columns=[desc[0] for desc in cur.description],
        )
finally:
    conn.close()

display(
    year_statistics.set_index("Year")
    .style.background_gradient(subset=["Rows"], cmap="Blues")
    .background_gradient(subset=["Approval %"], cmap="RdYlGn", vmin=20, vmax=55)
    .format(
        {
            "Rows": "{:,}",
            "Approved": "{:,}",
            "Rejected": "{:,}",
            "Approval %": "{:.1f}%",
        }
    )
    .set_caption("Table 2:  Row count and approval distribution per year (live query)")
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("font-size", "0.95em"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("padding-bottom", "8px"),
                ],
            }
        ]
    )
)

,Rows,Approved,Rejected,Withdrawn,Approval %
Year,,,,,
2018,"1,105",496,609,0,44.9%
2019,"1,237",482,755,0,39.0%
2020,"1,482",484,998,0,32.7%
2021,"1,238",426,812,0,34.4%
2022,"1,207",420,787,0,34.8%
2023,"1,258",477,781,0,37.9%
2024,"1,272",344,928,0,27.0%
2025,"1,184",462,721,1,39.0%


In [8]:
# Null audit - verify that core non-nullable columns have zero nulls in the DB.
non_null_checks = [
    "source_year",
    "diarienummer",
    "utbildningsnamn",
    "beslut",
    "beslut_normalized",
    "lan",
]

null_audit_rows = []
conn = psycopg2.connect(**PG_DSN)
try:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM applications")
        total_rows = cur.fetchone()[0]
        for column_name in non_null_checks:
            if not re.fullmatch(r"[A-Za-z0-9_]+", column_name):
                raise ValueError(f"Unsafe column name: '{column_name}'")
            cur.execute(
                f"SELECT COUNT(*) FROM applications WHERE {column_name} IS NULL"
            )
            null_count = cur.fetchone()[0]
            null_audit_rows.append(
                {
                    "Column": column_name,
                    "Expected nulls": 0,
                    "Actual nulls": null_count,
                    "Status": "Pass" if null_count == 0 else "Fail",
                }
            )
finally:
    conn.close()

null_audit_dataframe = pd.DataFrame(null_audit_rows).set_index("Column")


def _style_audit_status(value):
    if value == "Pass":
        return "background-color: rgba(46, 204, 113, 0.25); font-weight: 600"
    if value == "Fail":
        return "background-color: rgba(231, 76, 60, 0.25); font-weight: 600"
    return ""


display(
    null_audit_dataframe.style.map(_style_audit_status, subset=["Status"])
    .format({"Actual nulls": "{:,}"})
    .set_caption("Table 3 - Null audit on core non-nullable columns")
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("font-size", "0.95em"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("padding-bottom", "8px"),
                ],
            }
        ]
    )
)
print(f"Total rows in table: {total_rows:,}")

for audit_row in null_audit_rows:
    log_event(
        "verify",
        "ok" if audit_row["Status"] == "Pass" else "error",
        f"Null check: {audit_row['Column']}",
        "all non-null"
        if audit_row["Actual nulls"] == 0
        else f"{audit_row['Actual nulls']} nulls found",
    )

,Expected nulls,Actual nulls,Status
Column,,,
source_year,0,0,Pass
diarienummer,0,0,Pass
utbildningsnamn,0,0,Pass
beslut,0,0,Pass
beslut_normalized,0,0,Pass
lan,0,0,Pass


Total rows in table: 9,983


In [ ]:
# Value distribution for the two most important categorical columns.
conn = psycopg2.connect(**PG_DSN)
try:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                beslut_normalized AS "Decision",
                COUNT(*) AS "Rows",
                ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS "Share %"
            FROM applications
            GROUP BY beslut_normalized
            ORDER BY "Rows" DESC
        """)
        beslut_distribution = pd.DataFrame(
            cur.fetchall(), columns=[d[0] for d in cur.description]
        )
        cur.execute("""
            SELECT
                COALESCE(studieform, '[missing]') AS "Study form",
                COUNT(*) AS "Rows",
                ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS "Share %"
            FROM applications
            GROUP BY studieform
            ORDER BY "Rows" DESC
        """)
        studieform_distribution = pd.DataFrame(
            cur.fetchall(), columns=[d[0] for d in cur.description]
        )
finally:
    conn.close()

caption_style = [
    {
        "selector": "caption",
        "props": [
            ("font-size", "0.95em"),
            ("font-weight", "bold"),
            ("text-align", "left"),
            ("padding-bottom", "8px"),
        ],
    }
]

display(
    beslut_distribution.set_index("Decision")
    .style.background_gradient(subset=["Rows"], cmap="Blues")
    .format({"Rows": "{:,}", "Share %": "{:.1f}%"})
    .set_caption("Table 4 , beslut_normalized distribution")
    .set_table_styles(caption_style)
)
display(
    studieform_distribution.set_index("Study form")
    .style.background_gradient(subset=["Rows"], cmap="Blues")
    .format({"Rows": "{:,}", "Share %": "{:.1f}%"})
    .set_caption("Table 5 , Study form distribution")
    .set_table_styles(caption_style)
)

,Rows,Share %
Decision,,
rejected,"6,391",64.0%
approved,"3,591",36.0%
withdrawn,1,0.0%


,Rows,Share %
Study form,,
Bunden,"5,722",57.3%
Distans,"4,261",42.7%


---
## 4. Notes

- **Idempotent load:** `DROP TABLE IF EXISTS` in the schema means re-running this notebook always produces a clean, consistent table.
- **No transformation here:** All harmonization, cleaning, and enrichment is handled in `data_preparation.ipynb`. This notebook only transports data from CSV to PostgreSQL.
- **Standalone equivalent:** `db/load.py` replicates this notebook as a script , useful for automated reloads without Jupyter.
- **Next step:** The FastAPI layer (`api/main.py`) reads directly from this table.

---
## Step 10: FastAPI

The `applications` table is exposed as a REST API via three files:

| File | Role |
|:---|:---|
| `api/models.py` | Pydantic response models (`ApplicationRow`, `YearStats`, `PredictRequest`, `PredictResponse`) |
| `api/database.py` | psycopg2 connection pool, all query functions, and the `reload_database` helper |
| `api/main.py` | FastAPI app, route definitions, filter logic, CSV export, and ML predict endpoint |

**Endpoints (17 total)**

| Method | Path | Description |
|:---|:---|:---|
| `GET` | `/health` | Database connectivity and row-count check |
| `GET` | `/applications` | Filtered list: `year`, `decision`, `region`, `municipality`, `provider`, `studieform`; paginated |
| `GET` | `/applications/{diarienummer}` | All rows for a case number |
| `GET` | `/providers` | Unique provider names (alphabetical) |
| `GET` | `/providers/{name}/applications` | All applications by a provider |
| `GET` | `/stats/by-year` | Yearly totals and approval rates |
| `GET` | `/stats/by-education-area` | Grouped by `utbildningsomrade` |
| `GET` | `/stats/by-region` | Grouped by `lan` |
| `GET` | `/stats/by-study-form` | Grouped by `studieform` |
| `GET` | `/stats/by-provider` | Top providers by application count |
| `GET` | `/stats/multi-municipality` | Summary for multi-municipality applications |
| `GET` | `/stats/municipality-enrichment` | MYH metrics joined with SCB population and labour-market context |
| `GET` | `/export/applications` | CSV download (same filters as `/applications`) |
| `GET` | `/export/stats/by-year` | CSV download of yearly stats |
| `GET` | `/pipeline/status` | Live data quality checks against the `applications` table |
| `POST` | `/predict` | ML model: approval probability for a new application |
| `POST` | `/refresh` | Drop and reload the DB from CSV |

Start the server from the project root:

```
uvicorn api.main:app --reload
```

Interactive docs: `http://127.0.0.1:8000/docs`

### `api/models.py`

Pydantic response models. Field-level comments explain which columns are nullable and why (source-year gaps in the original Excel files).

In [10]:
# Display api/models.py with Python syntax highlighting
models_py = (BASE / "api" / "models.py").read_text(encoding="utf-8")
formatter = HtmlFormatter(style="monokai", noclasses=True)
highlighted = highlight(models_py, PythonLexer(), formatter)
highlighted = highlighted.replace("background: #272822", "background: #1a2535")
display(HTML(highlighted))

### `api/database.py`

psycopg2 connection pool and all query functions. Raw SQL with `%(name)s` parameters, no ORM. Inline comments explain ILIKE usage and the CSV type-conversion logic in `reload_database`.

In [11]:
# Display api/database.py with Python syntax highlighting
db_py = (BASE / "api" / "database.py").read_text(encoding="utf-8")
formatter = HtmlFormatter(style="monokai", noclasses=True)
highlighted = highlight(db_py, PythonLexer(), formatter)
highlighted = highlighted.replace("background: #272822", "background: #1a2535")
display(HTML(highlighted))

### `api/main.py`

FastAPI app with all route handlers. Every route function has a docstring that FastAPI surfaces as the endpoint description in `/docs`.

In [12]:
# Display api/main.py with Python syntax highlighting
main_py = (BASE / "api" / "main.py").read_text(encoding="utf-8")
formatter = HtmlFormatter(style="monokai", noclasses=True)
highlighted = highlight(main_py, PythonLexer(), formatter)
highlighted = highlighted.replace("background: #272822", "background: #1a2535")
display(HTML(highlighted))

---
## 5. Pipeline Report

A full record of every event logged during this run - one row per setup check, schema operation, data load, and verification step.
Green = ok, amber = warning, red = error.

This is the single place to look when something goes wrong. The pipeline keeps running past warnings and logs them here, so this table is always populated even after a partial failure.

In [13]:
if not PIPELINE_LOG:
    print("No events logged.")
else:
    log_df = pd.DataFrame(PIPELINE_LOG)

    ok_count = int((log_df["Level"] == "ok").sum())
    warn_count = int((log_df["Level"] == "warning").sum())
    err_count = int((log_df["Level"] == "error").sum())

    print(
        f"Pipeline complete  |  {ok_count} ok  |  {warn_count} warning(s)  |  {err_count} error(s)"
    )
    if err_count:
        print("  -> Action required: review the errors before using the database.")
    elif warn_count:
        print("  -> Warnings present: data may be usable but check the details.")
    else:
        print("  -> All events clean - database is ready for use.")
    print()

    def _style_level(value: str) -> str:
        if value == "ok":
            return "background-color: rgba(46, 204, 113, 0.18)"
        if value == "warning":
            return "background-color: rgba(230, 126, 34, 0.28); font-weight: 600"
        if value == "error":
            return "background-color: rgba(231, 76, 60, 0.28); font-weight: 600"
        return ""

    display(log_df.style.map(_style_level, subset=["Level"]))

Pipeline complete  |  14 ok  |  0 warning(s)  |  0 error(s)
  -> All events clean - database is ready for use.



,Step,Level,Message,Detail
0,setup,ok,CSV found,myh-data-pipeline\data\curated_applications.csv
1,setup,ok,schema.sql found,myh-data-pipeline\db\schema.sql
2,setup,ok,Database 'data_pipeline_de25' already exists,
3,setup,ok,Database reachable: data_pipeline_de25,
4,schema,ok,Schema applied (DROP + CREATE + indexes),
5,schema,ok,"Verified: 27 columns, 6 indexes",
6,load,ok,"CSV read: 9,983 rows, 26 columns",
7,insert,ok,"Bulk insert complete: 9,983 rows",
8,verify,ok,Null check: source_year,all non-null
9,verify,ok,Null check: diarienummer,all non-null
